# 02 · Evaluate, compare and explain

**AIN 320 final project, Phase 1 (PRD §6, Checklist Steps 5–6).**

| Step | Command it runs | Output (on Drive, `outputs/eval/test/`) |
|---|---|---|
| 1. Test metrics per model | `src.evaluate` | `metrics.json`, per-class table, confusion matrix, predictions |
| 2. DiT reference | `src.evaluate --dit` | same, for `microsoft/dit-base-finetuned-rvlcdip` |
| 3. Comparison tables | `src.compare` | M1 vs M2 vs M3 vs DiT, invoice table, literature table, accuracy-vs-speed plot |
| 4. Grad-CAM | `src.gradcam` | heatmaps for correct and wrong pages, incl. invoice |
| 5. Error analysis | `src.error_analysis` | ~50 misclassified pages for the team to label |
| 6. Duplicates (optional) | `src.duplicates` | accuracy with and without train/test near-duplicates |

**The test set is used once per final model.** `src.evaluate` refuses to re-run a model that already has test
results (it prints the stored numbers instead), so re-running this notebook is safe. Only evaluate models you
have finished tuning on the validation set.

**Runtime:** GPU (T4). Throughput numbers are only comparable when measured on the same GPU type, so run all
models in the same kind of session and report the GPU name.

## Settings

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"   # TODO: your team's GitHub repo
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/ain320_doccls"

RUNS = ["custom_cnn_main", "resnet50_main", "efficientnet_b0_main"]   # folders in outputs/runs/
INCLUDE_DIT = True
GRADCAM_CLASSES = ["invoice", "form", "questionnaire", "letter", "memo",
                   "scientific report", "budget", "news article"]
ERROR_RUN = None        # run whose errors the team analyses; None = best macro-F1
DUP_THRESHOLD = None    # set after looking at the duplicates sheet (step 6); None = no duplicate columns

## Setup: Drive, code, GPU, data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone the repo (first run) or pull the latest code, then install dependencies.
import os
REPO_DIR = "/content/final-project"
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
import os
import json
import torch
import pandas as pd
from pathlib import Path
from IPython.display import Image as ShowImage, Markdown, display

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("CUDA available:", torch.cuda.is_available())

WC_DRIVE = f"{DRIVE_ROOT}/working_copy"
WC_LOCAL = "/content/data/working_copy"
!python -m src.data.local_copy unpack --src {WC_DRIVE} --dst {WC_LOCAL}
!python -m src.data.local_copy verify --data-dir {WC_LOCAL} --lists test

os.environ["DOCCLS_DATA_DIR"] = WC_LOCAL
os.environ["DOCCLS_OUTPUT_DIR"] = f"{DRIVE_ROOT}/outputs"
OUT = Path(DRIVE_ROOT) / "outputs"
EVAL = OUT / "eval" / "test"

## 1 · Test-set metrics for each trained model

Accuracy, macro-F1, per-class P/R/F1, invoice P/R/F1, 16×16 confusion matrix, parameters, model size and
throughput (model only, and end to end including image loading).

In [ ]:
for run in RUNS:
    run_dir = OUT / "runs" / run
    if not (run_dir / "best.pt").exists():
        print(f"skipping {run}: no best.pt yet (train it with 01_train.ipynb)")
        continue
    !python -m src.evaluate --run-dir {run_dir}

## 2 · DiT reference (evaluation only)

`microsoft/dit-base-finetuned-rvlcdip`, a document-pretrained transformer trained by its authors on the full
320k training set. It is run with its own preprocessing on our test copy and is **not** trained by us. If this
cell fails (download or memory), set `INCLUDE_DIT = False` and cite the published number instead.

In [ ]:
if INCLUDE_DIT:
    !python -m src.evaluate --dit

## 3 · Comparison tables

The tables are written as Markdown, ready to paste into the report. Literature numbers come from
`results/published_results.yaml` and are marked **(unverified)** until someone checks each against its paper.

In [ ]:
DUP_FLAG = f"--dup-threshold {DUP_THRESHOLD}" if DUP_THRESHOLD is not None else ""
!python -m src.compare {DUP_FLAG} > /dev/null
CMP = EVAL / "comparison"
for name in ("comparison", "invoice", "literature"):
    display(Markdown(f"### {name}\n\n" + (CMP / f"{name}.md").read_text()))
display(ShowImage(str(CMP / "accuracy_vs_speed.png"), width=640))

### Confusion matrices and most-confused pairs

In [ ]:
for metrics_file in sorted(EVAL.glob("*/metrics.json")):
    m = json.loads(metrics_file.read_text())
    display(Markdown(f"#### {m['role']} {m['name']}: accuracy {100 * m['accuracy']:.2f}%, macro-F1 {m['macro_f1']:.4f}"))
    display(ShowImage(str(metrics_file.parent / "confusion_matrix.png"), width=720))
    display(pd.DataFrame(m["top_confusions"]).head(8))

## 4 · Grad-CAM: what do the models look at?

For each class (invoice always included): pages the model got right (top row) and wrong (bottom row).
Orange marks the regions that drove the predicted class. Look for layout cues (tables, headers, totals,
letterheads). Heat on borders, margins or scan noise is a sign of shortcut learning and goes in the report.

In [ ]:
CLASS_ARGS = " ".join(f'"{c}"' for c in GRADCAM_CLASSES)
for run in RUNS:
    if (EVAL / run / "predictions.csv").exists():
        !python -m src.gradcam --run-dir {OUT / "runs" / run} --classes {CLASS_ARGS}

In [ ]:
for run in RUNS:
    folder = EVAL / run / "gradcam"
    if folder.exists():
        display(Markdown(f"#### {run}"))
        display(ShowImage(str(folder / "gradcam_overview.png")))
        display(ShowImage(str(folder / "gradcam_invoice.png")))

## 5 · Error analysis (~50 misclassified pages)

1. The cell below samples 50 errors of one model and writes contact sheets plus `errors_to_label.csv`.
2. Open the CSV (e.g. in Google Sheets) and fill the `category` column for every row:
   `ambiguous` (either class is defensible), `label_noise` (the dataset label is wrong) or
   `model_error` (label right, model wrong). Add `notes` where useful.
3. Save it as `errors_labelled.csv` in the same folder and run the summary cell.

In [ ]:
if ERROR_RUN is None:
    table = pd.read_csv(EVAL / "comparison" / "comparison.csv")
    ERROR_RUN = table[table.role != "Ref"].sort_values("macro-F1").iloc[-1]["model"]
print("error analysis for", ERROR_RUN)
!python -m src.error_analysis sample --eval-dir {EVAL / ERROR_RUN} --n 50
ERR_DIR = EVAL / ERROR_RUN / "error_analysis"
for sheet in sorted(ERR_DIR.glob("errors_sheet_*.png"))[:1]:
    display(ShowImage(str(sheet)))
print("all sheets and the CSV to fill in:", ERR_DIR)

In [ ]:
LABELLED = EVAL / ERROR_RUN / "error_analysis" / "errors_labelled.csv"
if LABELLED.exists():
    !python -m src.error_analysis summarize --csv {LABELLED}
else:
    print(f"Not labelled yet: fill in the categories and save as {LABELLED}")

## 6 · Optional: train/test near-duplicates

RVL-CDIP has documents that appear in both train and test, which inflates test scores (PRD §4.4). This finds
each test page's most similar page in our Main training subset (perceptual hash; a few minutes).
Look at the sheet: each row is a distance range. Choose the largest distance where pairs are still clearly
the same document, set `DUP_THRESHOLD` in the settings cell, and re-run section 3.

In [ ]:
!python -m src.duplicates find --train-list main_train
!python -m src.duplicates sheet
display(ShowImage(str(EVAL / "duplicates_sheet.png")))

## 7 · Download the results for the repo

Everything above is already saved on Drive. This bundles the small files (tables, metrics, figures) so you can
copy them into the repo's `results/` folder and commit them. Checkpoints stay on Drive.

In [ ]:
import shutil
bundle = shutil.make_archive("/content/eval_results", "zip", root_dir=EVAL)
print(bundle, f"{os.path.getsize(bundle) / 1e6:.1f} MB")
from google.colab import files
files.download(bundle)